# Feature Stores and Data Validation for ML

Notebook for **Chapter 5** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch05.html

Tip: change a number, predict the output, then run the cell again.

## 5.1 Feature Store Concepts: Offline, Online, Reuse

In [ ]:
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Int64
from datetime import timedelta
user = Entity(name="user", join_keys=["user"])
src = FileSource(path="data/user_features.parquet", timestamp_field="ts")
user_stats = FeatureView(name="user_stats", entities=[user], ttl=timedelta(days=30),
                         schema=[Field(name="orders_30d", dtype=Int64)], source=src)
# training:  store.get_historical_features(entity_df, ["user_stats:orders_30d"])
# serving:   store.get_online_features(features=["user_stats:orders_30d"], entity_rows=[{"user": "u1"}])

In [ ]:
import pandas as pd
offline = pd.DataFrame({
 "user": ["u1","u1","u1","u2","u2"],
 "ts": pd.to_datetime(["2026-01-05","2026-01-12","2026-01-25","2026-01-01","2026-01-18"]),
 "orders_30d": [1,2,9,0,7]})
latest = offline.sort_values("ts").groupby("user").tail(1)
online = {r.user: {"orders_30d": int(r.orders_30d), "ts": r.ts} for r in latest.itertuples()}
now = pd.Timestamp("2026-02-20"); ttl = pd.Timedelta(days=30)
def get(user):
    row = online.get(user)
    if row is None: return "MISSING -> use default"
    age = now - row["ts"]
    return f"{row['orders_30d']} (age {age.days} d)" + ("  STALE" if age > ttl else "")
for u in ["u1","u2","u3"]: print(u, "->", get(u))

## 5.2 Point-in-Time Correctness (Avoiding Leakage)

In [ ]:
import pandas as pd
events = pd.DataFrame({
 "user": ["u1","u1","u2"],
 "ts": pd.to_datetime(["2026-01-10","2026-01-20","2026-01-15"]),
 "bought": [0,1,1]})
feat = pd.DataFrame({
 "user": ["u1","u1","u1","u2","u2"],
 "ts": pd.to_datetime(["2026-01-05","2026-01-12","2026-01-25","2026-01-01","2026-01-18"]),
 "orders_30d": [1,2,9,0,7]})
bad = events.merge(feat.sort_values("ts").groupby("user").tail(1)[["user","orders_30d"]], on="user")
print("LEAKY join (latest value):"); print(bad.to_string(index=False))
good = pd.merge_asof(events.sort_values("ts"), feat.sort_values("ts"), on="ts", by="user", direction="backward")
print("\nPOINT-IN-TIME join:"); print(good.to_string(index=False))

## 5.3 Data Validation for ML

In [ ]:
import pandas as pd
df = pd.DataFrame({"age":[25,40,None,200,33],"country":["IN","IN","US","XX","IN"],"label":[0,1,1,0,2]})
schema = {"age":{"min":0,"max":120,"nullable":False},"country":{"allowed":{"IN","US","UK"}},"label":{"allowed":{0,1}}}
errs=[]
for col,r in schema.items():
    s=df[col]
    if "nullable" in r and not r["nullable"] and s.isna().any(): errs.append(f"{col}: {int(s.isna().sum())} null(s)")
    if "min" in r:
        bad=s[(s<r["min"])|(s>r["max"])]
        if len(bad): errs.append(f"{col}: out of range {bad.tolist()}")
    if "allowed" in r:
        bad=s[~s.isin(r["allowed"])]
        if len(bad): errs.append(f"{col}: unexpected values {bad.tolist()}")
print("\n".join(errs))

In [ ]:
# conceptual example
validator.expect_column_values_to_be_between("age", min_value=0, max_value=120)
validator.expect_column_values_to_not_be_null("age")
validator.expect_column_values_to_be_in_set("country", ["IN", "US", "UK"])

## 5.4 Leakage at Scale, Skew and Streaming Features

Section 5.2 introduced point-in-time correctness with a tiny example. This section scales it up: you will measure how much a leak inflates a score, use the one-line pandas join that does it correctly, quantify a small definition mismatch between SQL and an API, build a streaming window feature by hand, and see the Feast commands that wrap all of this. The Python ran here. Feast needs its own install and a repository, so those commands are labelled not run here.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(11)
n_users, weeks = 4000, 20
rate = rng.gamma(4, 0.6, n_users)                         # each user's normal orders per week
churn_week = np.where(rng.random(n_users) < 0.3, rng.integers(11, 15, n_users), 999)

rows = []
for u in range(n_users):
    for w in range(weeks):
        r = rate[u]
        if churn_week[u] < 999 and w >= churn_week[u] - 3: r *= 0.5     # slowing down before leaving
        if w >= churn_week[u]: r = 0                                     # gone
        rows.append((u, w, rng.poisson(r)))
orders = pd.DataFrame(rows, columns=["user", "week", "orders"])
orders["orders_4w"] = orders.groupby("user")["orders"].transform(lambda s: s.rolling(4, min_periods=1).sum())

# Prediction date: end of week 10. Label: does the user churn in weeks 11 to 14?
T = 10
active = churn_week > T
labels = pd.DataFrame({"user": np.arange(n_users), "y": (churn_week <= 14).astype(int)})[active]

late = orders[orders.week == weeks - 1].set_index("user")["orders_4w"]      # latest value in the store today
asof = orders[orders.week == T].set_index("user")["orders_4w"]              # value known on the prediction date
df = labels.assign(naive=late.reindex(labels.user).values, pit=asof.reindex(labels.user).values)

train = df.user < 2800
def fit_eval(col_train, col_test):
    m = LogisticRegression().fit(df.loc[train, [col_train]].values, df.loc[train, "y"])
    return roc_auc_score(df.loc[~train, "y"], m.predict_proba(df.loc[~train, [col_test]].values)[:, 1])

print("rows:", len(df), " churn rate:", round(df.y.mean(), 3))
print("naive join, tested on naive:      AUC", round(fit_eval("naive", "naive"), 3))
print("naive-trained, live (point-in-time): AUC", round(fit_eval("naive", "pit"), 3))
print("point-in-time, trained and live:  AUC", round(fit_eval("pit", "pit"), 3))

rows: 4000  churn rate: 0.315
naive join, tested on naive:      AUC 0.996
naive-trained, live (point-in-time): AUC 0.635
point-in-time, trained and live:  AUC 0.635


In [ ]:
import pandas as pd
# merge_asof does the point-in-time join: for each event, the latest feature row at or before it
events = pd.DataFrame({"user": ["u1", "u1", "u2"], "event_ts": pd.to_datetime(["2026-01-20", "2026-02-03", "2026-01-30"]),
                       "label": [0, 1, 0]}).sort_values("event_ts")
feats = pd.DataFrame({"user": ["u1", "u1", "u1", "u2", "u2"],
                      "ts": pd.to_datetime(["2026-01-05", "2026-01-25", "2026-02-10", "2026-01-01", "2026-02-02"]),
                      "orders_30d": [1, 4, 9, 0, 7]}).sort_values("ts")
out = pd.merge_asof(events, feats, left_on="event_ts", right_on="ts", by="user",
                    direction="backward", tolerance=pd.Timedelta(days=30))
print(out[["user", "event_ts", "ts", "orders_30d", "label"]].to_string(index=False))

user   event_ts         ts  orders_30d  label
  u1 2026-01-20 2026-01-05           1      0
  u2 2026-01-30 2026-01-01           0      0
  u1 2026-02-03 2026-01-25           4      1


In [ ]:
from datetime import date, timedelta
# Training-serving skew from a one-day definition difference.
import random
random.seed(4)
order_days = {u: sorted(date(2026, 3, 1) + timedelta(days=random.randint(0, 59)) for _ in range(random.randint(0, 25))) for u in range(2000)}
today = date(2026, 4, 30)
def sql_version(u):            # "last 30 days": today - 30 < d <= today   (30 days)
    return sum(today - timedelta(days=30) < d <= today for d in order_days[u])
def api_version(u):            # off-by-one: includes the day 30 days ago  (31 days)
    return sum(today - timedelta(days=30) <= d <= today for d in order_days[u])
diff = [u for u in order_days if sql_version(u) != api_version(u)]
print("users whose feature differs:", len(diff), "of 2000 =", f"{len(diff)/2000:.1%}")
print("example:", [(u, sql_version(u), api_version(u)) for u in diff[:3]])

users whose feature differs: 410 of 2000 = 20.5%
example: [(0, 1, 2), (5, 6, 7), (8, 7, 8)]


In [ ]:
from collections import deque
from datetime import datetime, timedelta
# A streaming feature: orders in the last 10 minutes per user, with a sliding window.
events = [("u1", "12:00:30"), ("u1", "12:04:10"), ("u2", "12:05:00"), ("u1", "12:09:45"), ("u1", "12:11:00"), ("u1", "12:16:30")]
def t(s): return datetime.strptime("2026-05-01 " + s, "%Y-%m-%d %H:%M:%S")
window = timedelta(minutes=10)
buf = {}
for user, ts in events:
    q = buf.setdefault(user, deque()); q.append(t(ts))
    while q and q[0] <= t(ts) - window:      # drop events older than the window
        q.popleft()
    print(f"{ts} {user} -> orders_10m = {len(q)}")

12:00:30 u1 -> orders_10m = 1
12:04:10 u1 -> orders_10m = 2
12:05:00 u2 -> orders_10m = 1
12:09:45 u1 -> orders_10m = 3
12:11:00 u1 -> orders_10m = 3
12:16:30 u1 -> orders_10m = 3


In [ ]:
# feature_repo/features.py  (Feast; API names change between versions, check the docs of your version)
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

user = Entity(name="user", join_keys=["user_id"])
src = FileSource(path="data/user_stats.parquet", timestamp_field="event_ts")
user_stats = FeatureView(
    name="user_stats", entities=[user], ttl=timedelta(days=2),
    schema=[Field(name="orders_30d", dtype=Int64), Field(name="avg_basket", dtype=Float32)],
    source=src,
)

$ feast apply                                   # register the definitions
$ feast materialize-incremental 2026-05-01T00:00:00   # offline to online, only new data
$ python train.py   # calls store.get_historical_features(entity_df, features=[...])  point-in-time join
$ python serve.py   # calls store.get_online_features(entity_rows=[{"user_id": 7}], features=[...])


## Chapter recap

## Practice set

Problem 5.1 Offline or online store: a 3-year history of daily features for training? **Answer:** Offline.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch05.html